In [6]:
import pandas as pd
import numpy as np
import urllib.parse
import json

# ------------------------------
# Load data
# ------------------------------
query_file = "../data/qid_converted.csv"
metrics_file = "../data/network_metrics.csv"

query_df = pd.read_csv(query_file)
metrics_df = pd.read_csv(metrics_file)

# ------------------------------
# Title normalization (consistent with graph filtering)
# ------------------------------
def normalize_title(t):
    if pd.isna(t):
        return None
    t = urllib.parse.unquote(str(t))
    return t.strip().replace(" ", "_").lower()

query_df["title_clean"] = query_df["title"].apply(normalize_title)
metrics_df["title_clean"] = metrics_df["title_clean"].apply(normalize_title)

# ------------------------------
# Overlap check per language
# ------------------------------
print("\n🔍 Overlap check per language:")
for lang in sorted(query_df["language"].unique()):
    q_titles = set(query_df.loc[query_df["language"] == lang, "title_clean"])
    m_titles = set(metrics_df.loc[metrics_df["language"] == lang, "title_clean"])
    overlap = q_titles & m_titles
    print(f"{lang}: query={len(q_titles)}, metrics={len(m_titles)}, overlap={len(overlap)}")

# ------------------------------
# Merge datasets on qid + language
# ------------------------------
if "qid" not in query_df.columns:
    raise ValueError("❌ 'qid' column missing in query_converted.csv")

merged_df = query_df.merge(
    metrics_df,
    on=["language", "title_clean"],
    suffixes=("_query", "_metrics")
)

print(f"\n✅ Merged rows: {len(merged_df)} (before dropping duplicates)")

# ------------------------------
# Metrics to analyse
# ------------------------------
metrics_list = ["pagerank", "in_degree", "out_degree",
                "betweenness", "eigenvector", "clustering"]

# ------------------------------
# Normalisation (0–1 per language)
# ------------------------------
for metric in metrics_list:
    merged_df[f"{metric}_norm_perlang"] = merged_df.groupby("language")[metric] \
        .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)

# ------------------------------
# Per-QID bias computation
# ------------------------------
total_languages = metrics_df["language"].nunique()
bias_records = []

for qid, group in merged_df.groupby("qid"):
    present_langs = sorted(set(group["language"]))
    if not present_langs:
        continue

    en_title_row = group[group["language"] == "en"]
    en_title = en_title_row["title_clean"].iloc[0] if not en_title_row.empty else None
    coverage_bias = len(present_langs) / total_languages

    ranks_dict = {}
    for metric in metrics_list:
        sorted_langs = (
            group[["language", metric]]
            .sort_values(metric, ascending=False)["language"]
            .tolist()
        )
        ranks_dict[f"{metric}_rank_perqid"] = {lang: rank+1 for rank, lang in enumerate(sorted_langs)}

    bias_records.append({
        "qid": qid,
        "en_title": en_title,
        "langs_present": ",".join(present_langs),
        "coverage_bias": coverage_bias,
        **{f"{metric}_rank_perqid": json.dumps(ranks_dict[f"{metric}_rank_perqid"]) for metric in metrics_list}
    })

bias_df = pd.DataFrame(bias_records)
bias_df.to_csv("../data/bias_metrics.csv", index=False)

# ------------------------------
# Per-language normalised file
# ------------------------------
perlang_df = merged_df[["qid", "language"] + [f"{m}_norm_perlang" for m in metrics_list]].drop_duplicates()
perlang_df.to_csv("../data/bias_metrics_perlang.csv", index=False)

print(f"\n💾 bias_metrics.csv saved: {bias_df.shape[0]} QIDs")
print(f"💾 bias_metrics_perlang.csv saved: {perlang_df.shape[0]} rows")



🔍 Overlap check per language:
de: query=900, metrics=142107, overlap=900
en: query=1291, metrics=407236, overlap=1291
hi: query=528, metrics=25089, overlap=528
ja: query=731, metrics=142798, overlap=731

✅ Merged rows: 3627 (before dropping duplicates)

💾 bias_metrics.csv saved: 1985 QIDs
💾 bias_metrics_perlang.csv saved: 3607 rows


In [11]:
import os
import re
import pandas as pd

DATA_PATH = "../data"
BIAS_FILE = os.path.join(DATA_PATH, "bias_metrics.csv")

# Property files you already created
FILES = {
    "occupation": os.path.join(DATA_PATH, "occupation_mapped.csv"),   # expects occupation_category (or occupationLabel)
    "instance":   os.path.join(DATA_PATH, "instance_mapped.csv"),     # expects instance_category (or instanceOfLabel)
    "citizenship":os.path.join(DATA_PATH, "citizenship.csv"), # expects citizenshipLabel
    "origin":     os.path.join(DATA_PATH, "origin.csv"),      # expects originLabel
}

# Which columns to keep in the final per-property CSV
BASE_KEEP = [
    "qid","en_title","langs_present","coverage_bias",
    # keep rank json columns if present
    "pagerank_rank_perqid","in_degree_rank_perqid","out_degree_rank_perqid",
    "betweenness_rank_perqid","eigenvector_rank_perqid","clustering_rank_perqid"
]

def norm_qid(val):
    if pd.isna(val): return None
    m = re.search(r'Q\d+', str(val), flags=re.IGNORECASE)
    return m.group(0).upper() if m else None

def load_bias(path):
    df = pd.read_csv(path)
    if "qid" not in df.columns:
        raise ValueError(" 'qid' column missing in bias_metrics.csv")
    df["qid"] = df["qid"].apply(norm_qid)
    # drop rows with missing qid just in case
    df = df.dropna(subset=["qid"]).drop_duplicates(subset=["qid"])
    return df

def load_prop(path):
    if not os.path.exists(path):
        print(f" Missing: {path}")
        return None
    df = pd.read_csv(path)

    # figure out qid column name
    qid_col = None
    for cand in ["qid","item","entity","id"]:
        if cand in df.columns:
            qid_col = cand
            break
    if qid_col is None:
        print(f" No qid-like column in {os.path.basename(path)}; columns={df.columns.tolist()}")
        return None

    df["qid"] = df[qid_col].apply(norm_qid)
    df = df.dropna(subset=["qid"]).drop_duplicates(subset=["qid"])

    return df

def pick_prop_column(propname, df):
    """Choose the best column to represent this property."""
    if propname == "occupation":
        for col in ["occupation_category","occupationLabel"]:
            if col in df.columns: return col
    if propname == "instance":
        for col in ["instance_category","instanceOfLabel"]:
            if col in df.columns: return col
    if propname == "citizenship":
        for col in ["citizenshipLabel"]:
            if col in df.columns: return col
    if propname == "origin":
        for col in ["originLabel"]:
            if col in df.columns: return col
    # fallback: try to guess
    candidates = [c for c in df.columns if c not in ["qid","item","itemLabel","instanceOfLabel"]]
    return candidates[0] if candidates else None

bias = load_bias(BIAS_FILE)
print(f" Loaded bias metrics: {bias.shape[0]} rows, {bias['qid'].nunique()} unique QIDs")

# ensure BASE_KEEP exists in bias
keep_cols_present = [c for c in BASE_KEEP if c in bias.columns]
if "en_title" not in keep_cols_present and "en_title" in bias.columns:
    keep_cols_present.append("en_title")

for propname, path in FILES.items():
    prop_df = load_prop(path)
    if prop_df is None:
        continue

    prop_col = pick_prop_column(propname, prop_df)
    if prop_col is None:
        print(f" Could not find a property column for {propname} in {os.path.basename(path)}")
        continue

    # Keep only what's needed from the property table
    slim_prop = prop_df[["qid", prop_col]].copy()

    # Diagnostics
    inter = set(bias["qid"]).intersection(set(slim_prop["qid"]))
    print(f"\n {propname}: bias QIDs={bias['qid'].nunique()}, prop QIDs={slim_prop['qid'].nunique()}, overlap={len(inter)}")

    # Merge (left join keeps all bias rows, we’ll drop NaNs after to keep only rows with that property)
    merged = bias.merge(slim_prop, on="qid", how="left")
    merged = merged.dropna(subset=[prop_col])

    # Keep only a clean subset of columns for plotting
    out_cols = list(dict.fromkeys(["qid"] + keep_cols_present + [prop_col]))  # preserve order, drop dups
    out = merged[[c for c in out_cols if c in merged.columns]].copy()

    out_path = os.path.join(DATA_PATH, f"bias_metrics_{propname}.csv")
    out.to_csv(out_path, index=False)
    print(f" Saved {propname}: {out.shape[0]} rows → {out_path}")

print("\n Done. You now have separate, clean bias files per property.")


 Loaded bias metrics: 1985 rows, 1985 unique QIDs

 occupation: bias QIDs=1985, prop QIDs=374, overlap=374
 Saved occupation: 374 rows → ../data/bias_metrics_occupation.csv

 instance: bias QIDs=1985, prop QIDs=1813, overlap=1813
 Saved instance: 1031 rows → ../data/bias_metrics_instance.csv

 citizenship: bias QIDs=1985, prop QIDs=472, overlap=472
 Saved citizenship: 472 rows → ../data/bias_metrics_citizenship.csv

 origin: bias QIDs=1985, prop QIDs=133, overlap=133
 Saved origin: 133 rows → ../data/bias_metrics_origin.csv

 Done. You now have separate, clean bias files per property.


In [9]:
import pandas as pd

# Load files
bias_df = pd.read_csv("../data/bias_metrics.csv")
mapped_df = pd.read_csv("../data/mapped_categories1.csv")

# Prepare mapping dataframe
mapped_df["itemLabel"] = mapped_df["itemLabel"].str.replace(" ", "_")
mapped_df.rename(columns={"itemLabel": "en_title"}, inplace=True)

# Merge
merged_df = pd.merge(
    bias_df,
    mapped_df[["en_title", "cultural_category"]],
    on="en_title",
    how="left"
)

# Drop missing categories
merged_df = merged_df.dropna(subset=["cultural_category"])
merged_df = merged_df[merged_df["cultural_category"].str.strip() != ""]

# Save updated filtered file
merged_df.to_csv("../data/bias_metrics_with_categories_filtered.csv", index=False)

print("Updated filtered QID-level data saved!")
print(merged_df["cultural_category"].value_counts())


Updated filtered QID-level data saved!
cultural_category
Arts & Entertainment            162
Literature & Philosophy          73
Sports & Athletics               55
Science & Engineering            38
Technology & Media               34
Humanities & Social Sciences     29
Politics & Governance            23
Economics & Business              7
Name: count, dtype: int64


In [11]:
import pandas as pd

# Load files
perlang_df = pd.read_csv("../data/bias_metrics_perlang.csv")
qid_df = pd.read_csv("../data/bias_metrics.csv")  # This contains qid + en_title
mapped_df = pd.read_csv("../data/mapped_categories1.csv")

# Prepare mapping dataframe
mapped_df["itemLabel"] = mapped_df["itemLabel"].str.replace(" ", "_")
mapped_df.rename(columns={"itemLabel": "en_title"}, inplace=True)

# Step 1: Add en_title to per-language data by merging with bias_metrics.csv
perlang_df = perlang_df.merge(
    qid_df[["qid", "en_title"]],
    on="qid",
    how="left"
)

# Step 2: Merge with mapped categories on en_title
perlang_df = perlang_df.merge(
    mapped_df[["en_title", "cultural_category"]],
    on="en_title",
    how="left"
)

# Step 3: Drop missing/empty categories
perlang_df = perlang_df.dropna(subset=["cultural_category"])
perlang_df = perlang_df[perlang_df["cultural_category"].str.strip() != ""]

# Save updated file
perlang_df.to_csv("../data/bias_metrics_perlang_with_categories_filtered.csv", index=False)

print("✅ Updated filtered per-language data saved!")
print(perlang_df["cultural_category"].value_counts())


✅ Updated filtered per-language data saved!
cultural_category
Arts & Entertainment            972
Literature & Philosophy         438
Sports & Athletics              330
Science & Engineering           228
Technology & Media              204
Humanities & Social Sciences    174
Politics & Governance           138
Economics & Business             42
Name: count, dtype: int64


In [12]:
import pandas as pd
import urllib.parse
import os

# Paths 
DATA_DIR = "../data"
query_file = os.path.join(DATA_DIR, "qid_converted.csv")
metrics_file = os.path.join(DATA_DIR, "network_metrics.csv")
out_file = os.path.join(DATA_DIR, "bias_metrics_perlang.csv")

# Load 
query_df = pd.read_csv(query_file)
metrics_df = pd.read_csv(metrics_file)

# Title cleaning 
def normalize_title(t):
    if pd.isna(t):
        return None
    return urllib.parse.unquote(str(t)).replace(" ", "_")

query_df["title_clean"] = query_df["title"].apply(normalize_title)
metrics_df["title_clean"] = metrics_df["title_clean"].apply(normalize_title)

# Merge on language + title_clean 
merged_df = query_df.merge(
    metrics_df,
    on=["language", "title_clean"],
    suffixes=("_query", "_metrics")
)

# Metrics to normalise
metrics_list = ["pagerank", "in_degree", "out_degree", "betweenness", "eigenvector", "clustering"]

# ==== Normalise each metric within language ====
for metric in metrics_list:
    merged_df[f"{metric}_norm_perlang"] = merged_df.groupby("language")[metric] \
        .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)

# ==== Keep only needed columns ====
perlang_df = merged_df[["qid", "language"] + [f"{m}_norm_perlang" for m in metrics_list]].drop_duplicates()

# ==== Save ====
perlang_df.to_csv(out_file, index=False)
print(f" bias_metrics_perlang.csv saved: {perlang_df.shape[0]} rows, {perlang_df['language'].nunique()} languages")


 bias_metrics_perlang.csv saved: 3450 rows, 4 languages
